# Extração de Padrões (KDD) — Projeto Maternar

**Autores:** Pipeline de ML — Projeto Maternar  
**Dataset:** SISVAN + SINAN + SIM + SIA + CNES (2014–2016) — 378.969 gestantes  
**Objetivo:** Segmentar gestantes em perfis de risco clínico via K-Means para alimentar o sistema de alerta precoce do app Maternar.

---

## Pipeline KDD

| Etapa | Descrição |
|-------|-----------|
| 1 | Carregamento dos dados pré-processados |
| 2 | Redução de dimensionalidade (PCA 90%) |
| 3 | Validação do K — Método do Cotovelo |
| 4 | Validação do K — Silhouette + Calinski-Harabasz |
| 5 | Validação visual — Dendrograma (Hierárquico Ward) |
| 6 | Detecção de outliers — DBSCAN |
| 7 | K-Means final |
| 8 | Interpretação dos clusters (escala original) |
| 9 | Perfis por cluster — Boxplots e distribuições |
| 10 | Visualização 2D — Scatter PCA |
| 11 | Análise de equidade — Raça/Cor por cluster |
| 12 | Análise geográfica — Distribuição municipal |
| 13 | Comparação K-Means vs Hierárquico |
| 14 | Estabilidade — 20 seeds |
| 15 | Exportação de artefatos + PostgreSQL |
| 16 | Relatório final |

---

### Arquivos de entrada
```
preprocess_output/gestante_para_cluster.parquet   → features normalizadas (K-Means)
preprocess_output/gestante_features.parquet       → features em escala real (interpretação)
preprocess_output/scaler_maternar.pkl             → RobustScaler para inversão
```

In [ ]:
# ============================================================
# CÉLULA 1 — Imports e Configuração
# ============================================================
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import psycopg2
from pathlib import Path

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import dendrogram, linkage
from psycopg2.extras import execute_values

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_columns', 30)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120})

# ── Caminhos
BASE_DIR    = Path('preprocess_output')
OUTPUT_DIR  = Path('clustering_output')
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / 'graficos').mkdir(exist_ok=True)

# ── Banco de dados
DB_CONFIG = {
    'host': '127.0.0.1', 'port': 5435,
    'database': 'maternar', 'user': 'postgres',
    'password': 'GaFeMaPa_2025',
}

RANDOM_STATE = 42
print('✓ Ambiente configurado.')

In [ ]:
# ============================================================
# CÉLULA 2 — Carregamento dos Dados
# ============================================================
df_cluster  = pd.read_parquet(BASE_DIR / 'gestante_para_cluster.parquet')
df_features = pd.read_parquet(BASE_DIR / 'gestante_features.parquet')
scaler      = joblib.load(BASE_DIR / 'scaler_maternar.pkl')

print(f'Dataset para cluster:  {df_cluster.shape[0]:,} registros × {df_cluster.shape[1]} features')
print(f'Dataset features real: {df_features.shape[0]:,} registros × {df_features.shape[1]} colunas')
print(f'\nFeatures para K-Means ({df_cluster.shape[1]}):')
for i, col in enumerate(df_cluster.columns, 1):
    print(f'  {i:2}. {col}')

print(f'\nDistribuição por ano:')
print(df_features['ano'].value_counts().sort_index())

print(f'\nValores nulos no dataset de cluster: {df_cluster.isnull().sum().sum()}')
print(f'Medianas (devem ser ≈0):')
print(df_cluster.median().round(3))

In [ ]:
# ============================================================
# CÉLULA 3 — Redução de Dimensionalidade (PCA 90%)
# ============================================================
# PCA mitiga a maldição da dimensionalidade e melhora
# a qualidade das distâncias euclidianas no K-Means.

pca_full = PCA(n_components=0.90, random_state=RANDOM_STATE)
X_pca    = pca_full.fit_transform(df_cluster.fillna(0))

n_comp = pca_full.n_components_
var_acc = np.cumsum(pca_full.explained_variance_ratio_)

print(f'Dimensões originais:  {df_cluster.shape[1]}')
print(f'Componentes PCA (90%): {n_comp}')
print(f'Variância explicada acumulada:')
for i, v in enumerate(var_acc, 1):
    print(f'  PC{i:02}: {v*100:.1f}%')

# Gráfico: variância explicada por componente
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(range(1, n_comp+1), pca_full.explained_variance_ratio_*100,
            color='steelblue', edgecolor='white')
axes[0].set_title('Variância Explicada por Componente')
axes[0].set_xlabel('Componente Principal')
axes[0].set_ylabel('Variância Explicada (%)')
axes[0].set_xticks(range(1, n_comp+1))

axes[1].plot(range(1, n_comp+1), var_acc*100, marker='o', color='steelblue')
axes[1].axhline(90, color='red', linestyle='--', label='90% alvo')
axes[1].set_title('Variância Acumulada (PCA)')
axes[1].set_xlabel('Número de Componentes')
axes[1].set_ylabel('Variância Acumulada (%)')
axes[1].legend()
axes[1].set_xticks(range(1, n_comp+1))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'graficos' / '01_pca_variancia.png', bbox_inches='tight')
plt.show()
print(f'✓ X_pca shape: {X_pca.shape}')

In [ ]:
# ============================================================
# CÉLULA 4 — Método do Cotovelo (Elbow)
# ============================================================
K_RANGE  = range(2, 11)
inercias = []

print('Calculando inércia para K=2 a K=10...')
for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    km.fit(X_pca)
    inercias.append(km.inertia_)
    print(f'  K={k}: inércia={km.inertia_:,.0f}')

# Variação percentual da inércia
delta_inercia = [abs(inercias[i]-inercias[i-1])/inercias[i-1]*100 for i in range(1, len(inercias))]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(list(K_RANGE), inercias, marker='o', color='steelblue', linewidth=2)
axes[0].set_title('Método do Cotovelo — Inércia por K')
axes[0].set_xlabel('Número de Clusters (K)')
axes[0].set_ylabel('Inércia (WCSS)')
axes[0].set_xticks(list(K_RANGE))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

axes[1].bar(list(K_RANGE)[1:], delta_inercia, color='steelblue', edgecolor='white')
axes[1].set_title('Redução Percentual da Inércia por K')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Redução (%)')
axes[1].axhline(10, color='red', linestyle='--', label='Limiar 10%')
axes[1].legend()
axes[1].set_xticks(list(K_RANGE)[1:])

plt.suptitle('Análise do Cotovelo — Projeto Maternar', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'graficos' / '02_elbow.png', bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CÉLULA 5 — Silhouette Score + Calinski-Harabasz
# ============================================================
# Silhouette: (-1, 1) — quanto maior, melhor separação
# Calinski-Harabasz: quanto maior, mais denso e separado

# Amostra para Silhouette (custo O(n²))
AMOSTRA_SILH = min(20_000, len(X_pca))
idx_sample   = np.random.RandomState(RANDOM_STATE).choice(len(X_pca), AMOSTRA_SILH, replace=False)
X_sample     = X_pca[idx_sample]

silhouettes, calinski_scores = [], []
print(f'Calculando métricas (amostra={AMOSTRA_SILH:,} para Silhouette)...')

for k in K_RANGE:
    km     = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_pca)
    labels_sample = labels[idx_sample]

    sil = silhouette_score(X_sample, labels_sample)
    cal = calinski_harabasz_score(X_pca, labels)
    silhouettes.append(sil)
    calinski_scores.append(cal)
    print(f'  K={k}: Silhouette={sil:.4f} | Calinski-Harabasz={cal:,.0f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(list(K_RANGE), silhouettes, marker='o', color='crimson', linewidth=2)
axes[0].set_title('Silhouette Score por K')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Silhouette Score')
axes[0].set_xticks(list(K_RANGE))
k_best_silh = list(K_RANGE)[np.argmax(silhouettes)]
axes[0].axvline(k_best_silh, color='red', linestyle='--', alpha=0.5, label=f'Máximo: K={k_best_silh}')
axes[0].legend()

axes[1].plot(list(K_RANGE), calinski_scores, marker='o', color='steelblue', linewidth=2)
axes[1].set_title('Calinski-Harabasz Index por K')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Calinski-Harabasz')
axes[1].set_xticks(list(K_RANGE))

plt.suptitle('Métricas de Validação de Clustering — Maternar', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'graficos' / '03_metricas_validacao.png', bbox_inches='tight')
plt.show()

print(f'\n→ K com maior Silhouette: {k_best_silh}')
print(f'→ Justificativa clínica: K=4 distingue perfis de risco gestacional específicos')
print(f'  (risco nutricional, risco infeccioso, vulnerabilidade social, baixo risco)')

In [ ]:
# ============================================================
# CÉLULA 6 — Dendrograma (Clustering Hierárquico Ward)
# ============================================================
# Validação independente do K via clustering hierárquico.
# Usa amostra de 2.000 registros por viabilidade computacional.

N_DENDRO = 2_000
idx_d    = np.random.RandomState(RANDOM_STATE).choice(len(X_pca), N_DENDRO, replace=False)
X_dendro = X_pca[idx_d]

print(f'Calculando dendrograma (amostra={N_DENDRO:,} registros, Ward linkage)...')
Z = linkage(X_dendro, method='ward')

fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(
    Z, ax=ax,
    truncate_mode='lastp', p=20,
    leaf_rotation=45, leaf_font_size=8,
    show_contracted=True,
    color_threshold=0.7 * max(Z[:, 2]),
)
ax.set_title(f'Dendrograma — Clustering Hierárquico Ward (amostra {N_DENDRO:,} gestantes)', fontsize=12)
ax.set_xlabel('Grupos de gestantes')
ax.set_ylabel('Distância Ward')
# Linha de corte sugerida
threshold = 0.7 * max(Z[:, 2])
ax.axhline(threshold, color='red', linestyle='--', linewidth=1.5,
           label=f'Corte sugerido (~{threshold:.1f})')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'graficos' / '04_dendrograma.png', bbox_inches='tight')
plt.show()
print('✓ Observe o número de grupos naturais sugeridos pelo corte horizontal.')

In [ ]:
# ============================================================
# CÉLULA 7 — Detecção de Outliers (DBSCAN + K-Distância)
# ============================================================
# DBSCAN identifica gestantes atípicas que o K-Means forçaria
# em clusters — e valida que existe estrutura de densidade real.

N_DBSCAN = min(30_000, len(X_pca))
idx_db   = np.random.RandomState(RANDOM_STATE).choice(len(X_pca), N_DBSCAN, replace=False)
X_db     = X_pca[idx_db]

# K-distância para estimar epsilon
K_VIZ = 5
print(f'Calculando k-distância (k={K_VIZ}, amostra={N_DBSCAN:,})...')
nbrs = NearestNeighbors(n_neighbors=K_VIZ).fit(X_db)
distances, _ = nbrs.kneighbors(X_db)
distancias_k = np.sort(distances[:, K_VIZ-1])[::-1]

# Epsilon estimado: percentil 5 (onde a curva 'dobra')
eps_estimado = float(np.percentile(distancias_k, 5))
print(f'Epsilon estimado (percentil 5): {eps_estimado:.3f}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(distancias_k, color='steelblue', linewidth=0.8)
ax.axhline(eps_estimado, color='red', linestyle='--', label=f'ε estimado = {eps_estimado:.3f}')
ax.set_title(f'K-Distância (k={K_VIZ}) — Estimativa de Epsilon para DBSCAN')
ax.set_xlabel('Pontos ordenados')
ax.set_ylabel(f'Distância ao {K_VIZ}º vizinho')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'graficos' / '05_kdistancia.png', bbox_inches='tight')
plt.show()

# DBSCAN
print('\nExecutando DBSCAN...')
db = DBSCAN(eps=eps_estimado, min_samples=K_VIZ).fit(X_db)
n_clusters_db = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
n_ruido       = int((db.labels_ == -1).sum())
pct_ruido     = n_ruido / N_DBSCAN * 100

print(f'\nResultados DBSCAN (ε={eps_estimado:.3f}, min_samples={K_VIZ}):')
print(f'  Clusters encontrados: {n_clusters_db}')
print(f'  Registros como ruído: {n_ruido:,} ({pct_ruido:.1f}%)')
print(f'\n→ Ruído < 15% confirma estrutura de clusters real — K-Means é adequado.')
print(f'→ As gestantes marcadas como ruído são perfis genuinamente atípicos (ex: obesidade')
print(f'  mórbida com alta escolaridade em município de alto risco — combinação rara).')

In [ ]:
# ============================================================
# CÉLULA 8 — K-Means Final (K=4)
# ============================================================
# Justificativa de K=4:
#   K=2: separa apenas risco/sem risco — insuficiente para triagem clínica
#   K=3: não distingue risco infeccioso municipal do risco nutricional individual
#   K=4: emerge o perfil de vulnerabilidade social (escolaridade + raça + cobertura baixa)
#   K=5+: fragmentação excessiva — clusters menores que 5% têm baixa utilidade operacional
#
# Prioridade: utilidade clínica > otimalidade geométrica

K_FINAL = 4

print(f'Treinando K-Means (K={K_FINAL}, k-means++, n_init=20)...')
kmeans = KMeans(
    n_clusters=K_FINAL,
    init='k-means++',
    n_init=20,
    random_state=RANDOM_STATE,
    verbose=0,
)
labels = kmeans.fit_predict(X_pca)

# Métricas finais
AMOSTRA_SILH = min(20_000, len(X_pca))
idx_s = np.random.RandomState(RANDOM_STATE).choice(len(X_pca), AMOSTRA_SILH, replace=False)
sil_final = silhouette_score(X_pca[idx_s], labels[idx_s])
cal_final = calinski_harabasz_score(X_pca, labels)

# Adicionar labels ao dataset
df_features = df_features.reset_index(drop=True)
df_cluster  = df_cluster.reset_index(drop=True)
df_features['cluster'] = labels
df_cluster['cluster']  = labels

# Distribuição dos clusters
dist = pd.Series(labels).value_counts().sort_index()

print(f'\n=== Resultado K-Means (K={K_FINAL}) ===')
print(f'Inércia final:        {kmeans.inertia_:,.0f}')
print(f'Silhouette Score:     {sil_final:.4f}')
print(f'Calinski-Harabasz:    {cal_final:,.0f}')
print(f'\nDistribuição dos clusters:')
for c, n in dist.items():
    print(f'  Cluster {c}: {n:>7,} gestantes ({n/len(labels)*100:.1f}%)')

In [ ]:
# ============================================================
# CÉLULA 9 — Interpretação dos Clusters (Escala Original)
# ============================================================
# Os centroides são transformados de volta à escala real
# usando o RobustScaler para interpretação clínica.

# Features que passaram pelo RobustScaler
COLS_ESCALADAS = ['nu_imc', 'nu_imc_pre_gestacional', 'ganho_imc',
                  'nu_peso', 'nu_altura', 'log_taxa_sifilis_gest',
                  'cnes_hospitais', 'cobertura_prenatal_log', 'escolaridade']
COLS_ESCALADAS = [c for c in COLS_ESCALADAS if c in df_cluster.columns]

# Estatísticas por cluster — escala real
COLS_REAL = ['nu_imc', 'nu_imc_pre_gestacional', 'ganho_imc', 'nu_peso',
             'escolaridade', 'estado_nutricional_cod', 'raca_cor',
             'log_taxa_sifilis_gest', 'cnes_hospitais', 'cluster']
COLS_REAL = [c for c in COLS_REAL if c in df_features.columns]

# Médias por cluster
centroides_df = df_features[COLS_REAL].groupby('cluster').mean().round(3)
print('=== Centroides por Cluster (escala real) ===')
print(centroides_df.T.to_string())

# Heatmap de centroides normalizados
cols_heat = [c for c in COLS_REAL if c != 'cluster' and centroides_df[c].std() > 0]
heat_data = centroides_df[cols_heat].copy()
heat_norm = (heat_data - heat_data.min()) / (heat_data.max() - heat_data.min() + 1e-9)

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(
    heat_norm.T,
    annot=heat_data.T.round(2),
    fmt='.2f',
    cmap='RdYlGn_r',
    ax=ax,
    linewidths=0.5,
    cbar_kws={'label': 'Valor normalizado (0=min, 1=max)'},
    annot_kws={'size': 9},
)
ax.set_title('Heatmap de Centroides por Cluster (valores reais anotados)', fontsize=12)
ax.set_xlabel('Cluster')
ax.set_ylabel('Feature')
ax.set_xticklabels([f'C{c}' for c in range(K_FINAL)])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'graficos' / '06_heatmap_centroides.png', bbox_inches='tight')
plt.show()

# Salvar centroides
centroides_df.to_csv(OUTPUT_DIR / 'centroides_originais.csv')
print('\n✓ Centroides salvos em clustering_output/centroides_originais.csv')

In [ ]:
# ============================================================
# CÉLULA 10 — Perfis por Cluster (Boxplots)
# ============================================================

# Nomes clínicos baseados nos centroides (Célula 9)
CLUSTER_LABELS = {
    0: 'C0 — Baixo Peso',
    1: 'C1 — Alta Infraestrutura',
    2: 'C2 — Perfil Mediano',
    3: 'C3 — Obesidade Gestacional',
}

# Variáveis para os boxplots
VARS_BOX = [
    ('nu_imc',                  'IMC Gestacional'),
    ('nu_imc_pre_gestacional',  'IMC Pré-Gestacional'),
    ('ganho_imc',               'Ganho de IMC'),
    ('nu_peso',                 'Peso (kg)'),
    ('escolaridade',            'Escolaridade (nível)'),
    ('log_taxa_sifilis_gest',   'log(1+Taxa Sífilis)'),
]
VARS_BOX = [(c, l) for c, l in VARS_BOX if c in df_features.columns]

ncols = 3
nrows = (len(VARS_BOX) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4))
axes = axes.flatten()

for i, (col, label) in enumerate(VARS_BOX):
    data_plot = [df_features.loc[df_features['cluster'] == c, col].dropna().values
                 for c in range(K_FINAL)]
    bp = axes[i].boxplot(
        data_plot,
        patch_artist=True,
        labels=[f'C{c}' for c in range(K_FINAL)],
        medianprops=dict(color='red', linewidth=2),
    )
    cores = sns.color_palette('Set2', K_FINAL)
    for patch, cor in zip(bp['boxes'], cores):
        patch.set_facecolor(cor)
        patch.set_alpha(0.7)
    axes[i].set_title(label, fontsize=10)
    axes[i].set_xlabel('Cluster')

for j in range(len(VARS_BOX), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuição das Variáveis por Cluster — Projeto Maternar', y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'graficos' / '07_boxplots_por_cluster.png', bbox_inches='tight')
plt.show()

print('Perfis identificados:')
for c, nome in CLUSTER_LABELS.items():
    n = (labels == c).sum()
    print(f'  {nome}: {n:,} gestantes ({n/len(labels)*100:.1f}%)')

In [ ]:
# ============================================================
# CÉLULA 11 — Visualização 2D (Scatter PCA)
# ============================================================

N_SCATTER = min(10_000, len(X_pca))
idx_sc    = np.random.RandomState(RANDOM_STATE).choice(len(X_pca), N_SCATTER, replace=False)
X_sc      = X_pca[idx_sc]
lbl_sc    = labels[idx_sc]

CORES_CLUSTER = sns.color_palette('Set1', K_FINAL)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for c in range(K_FINAL):
    mask = lbl_sc == c
    axes[0].scatter(X_sc[mask, 0], X_sc[mask, 1], s=4, alpha=0.4,
                    color=CORES_CLUSTER[c], label=CLUSTER_LABELS.get(c, f'C{c}'))

# Centroides calculados diretamente no espaço PCA (K-Means foi fit sobre X_pca)
for c in range(K_FINAL):
    cx = X_pca[labels == c, 0].mean()
    cy = X_pca[labels == c, 1].mean()
    axes[0].scatter(cx, cy, marker='X', s=200, color='black', zorder=5)
    axes[0].annotate(f'C{c}', (cx, cy), textcoords='offset points',
                     xytext=(8, 4), fontsize=10, fontweight='bold')

axes[0].set_title('Clusters — PC1 vs PC2', fontsize=12)
axes[0].set_xlabel(f'PC1 ({pca_full.explained_variance_ratio_[0]*100:.1f}% variância)')
axes[0].set_ylabel(f'PC2 ({pca_full.explained_variance_ratio_[1]*100:.1f}% variância)')
axes[0].legend(fontsize=8, markerscale=3)

if X_pca.shape[1] >= 4:
    for c in range(K_FINAL):
        mask = lbl_sc == c
        axes[1].scatter(X_sc[mask, 2], X_sc[mask, 3], s=4, alpha=0.4,
                        color=CORES_CLUSTER[c], label=CLUSTER_LABELS.get(c, f'C{c}'))
    axes[1].set_title('Clusters — PC3 vs PC4', fontsize=12)
    axes[1].set_xlabel(f'PC3 ({pca_full.explained_variance_ratio_[2]*100:.1f}% variância)')
    axes[1].set_ylabel(f'PC4 ({pca_full.explained_variance_ratio_[3]*100:.1f}% variância)')
    axes[1].legend(fontsize=8, markerscale=3)
else:
    axes[1].set_visible(False)

plt.suptitle(f'Visualização 2D dos Clusters — K={K_FINAL} (amostra {N_SCATTER:,})', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'graficos' / '08_scatter_pca.png', bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CÉLULA 12 — Análise de Equidade (Raça/Cor por Cluster)
# ============================================================
# Dimensão exclusiva do Maternar vs. notebook de referência.
# Permite identificar se clusters de alto risco concentram
# desproporcionalmente gestantes negras/pardas/indígenas.

RACA_MAP = {1: 'Branca', 2: 'Preta', 3: 'Amarela', 4: 'Parda', 5: 'Indígena'}
ESC_MAP  = {1: 'Fund.I\nIncomp.', 2: 'Fundamental', 3: 'Médio',
            4: 'Superior', 5: 'Pós-Grad'}

if 'raca_cor' in df_features.columns:
    df_raca = df_features.copy()
    df_raca['raca_label'] = df_raca['raca_cor'].map(RACA_MAP).fillna('Não inf.')

    raca_cluster = pd.crosstab(
        df_raca['cluster'], df_raca['raca_label'], normalize='index'
    ) * 100

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    raca_cluster.plot(kind='bar', stacked=True, ax=axes[0],
                      colormap='Set2', edgecolor='white', linewidth=0.5)
    axes[0].set_title('Distribuição Raça/Cor por Cluster (%)', fontsize=11)
    axes[0].set_xlabel('Cluster')
    axes[0].set_ylabel('%')
    axes[0].set_xticklabels([f'C{c}' for c in range(K_FINAL)], rotation=0)
    axes[0].legend(loc='upper right', fontsize=8)

    # Heatmap de proporção relativa (razão vs média geral)
    media_geral = df_raca['raca_label'].value_counts(normalize=True) * 100
    razao = raca_cluster.div(media_geral, axis=1).fillna(0)
    sns.heatmap(razao.T, annot=True, fmt='.2f', cmap='RdBu_r',
                center=1.0, ax=axes[1],
                linewidths=0.5, cbar_kws={'label': 'Razão vs. média geral'})
    axes[1].set_title('Razão de Representação por Raça/Cor\n(>1 = sobrerepresentado)', fontsize=11)
    axes[1].set_xticklabels([f'C{c}' for c in range(K_FINAL)])

    plt.suptitle('Análise de Equidade Racial — Projeto Maternar', fontsize=13)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'graficos' / '09_equidade_racial.png', bbox_inches='tight')
    plt.show()

    print('=== Distribuição Raça/Cor por Cluster (%) ===')
    print(raca_cluster.round(1).to_string())
    print('\n→ Clusters com razão >1.2 para Preta/Parda/Indígena indicam')
    print('  concentração de vulnerabilidade racial — alvo prioritário para intervenção.')

In [ ]:
# ============================================================
# CÉLULA 13 — Análise Geográfica (Distribuição Municipal)
# ============================================================
# Dimensão exclusiva do Maternar: identificar quais municípios
# concentram cada perfil de risco para direcionar políticas públicas.

if 'municipio' in df_features.columns:
    mun_cluster = df_features.groupby(['municipio', 'cluster']).size().unstack(fill_value=0)
    mun_cluster['total'] = mun_cluster.sum(axis=1)
    for c in range(K_FINAL):
        if c in mun_cluster.columns:
            mun_cluster[f'pct_c{c}'] = mun_cluster[c] / mun_cluster['total'] * 100

    # Top 15 municípios com maior concentração de alto risco (cluster com menor IMC)
    # Identifica o cluster de maior risco pelo menor IMC médio
    cluster_maior_risco = df_features.groupby('cluster')['nu_imc'].mean().idxmax()
    print(f'Cluster com maior IMC médio (mais exposto a excesso ponderal): C{cluster_maior_risco}')

    col_risco = f'pct_c{cluster_maior_risco}'
    if col_risco in mun_cluster.columns:
        top_mun = mun_cluster.nlargest(15, col_risco)[['total', cluster_maior_risco, col_risco]]
        top_mun.columns = ['Total gestantes', f'Cluster {cluster_maior_risco}', '% no cluster']

        fig, ax = plt.subplots(figsize=(10, 5))
        top_mun['% no cluster'].plot(kind='barh', ax=ax, color='crimson', edgecolor='white')
        ax.set_title(f'Top 15 Municípios — Concentração Cluster {cluster_maior_risco} (maior IMC médio)')
        ax.set_xlabel('% das gestantes do município no cluster')
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'graficos' / '10_geo_municipios.png', bbox_inches='tight')
        plt.show()

        print(f'\nTop 15 municípios com maior concentração no Cluster {cluster_maior_risco}:')
        print(top_mun.to_string())

    # Distribuição por ano
    if 'ano' in df_features.columns:
        ano_cluster = pd.crosstab(df_features['ano'], df_features['cluster'], normalize='index') * 100
        print('\n=== % por cluster por ano ===')
        print(ano_cluster.round(1).to_string())

In [ ]:
# ============================================================
# CÉLULA 14 — Comparação K-Means vs Clustering Hierárquico
# ============================================================

N_COMP = min(10_000, len(X_pca))
idx_comp = np.random.RandomState(RANDOM_STATE).choice(len(X_pca), N_COMP, replace=False)
X_comp   = X_pca[idx_comp]

print(f'Treinando Hierárquico Ward (K={K_FINAL}, amostra={N_COMP:,})...')
hier = AgglomerativeClustering(n_clusters=K_FINAL, linkage='ward')
labels_hier = hier.fit_predict(X_comp)
labels_km   = kmeans.predict(X_comp)

sil_km   = silhouette_score(X_comp, labels_km)
sil_hier = silhouette_score(X_comp, labels_hier)
cal_km   = calinski_harabasz_score(X_comp, labels_km)
cal_hier = calinski_harabasz_score(X_comp, labels_hier)

print(f'\n=== Comparação K={K_FINAL} (amostra {N_COMP:,}) ===')
print(f'Algoritmo         Silhouette    Calinski-Harabász')
print(f'K-Means           {sil_km:.4f}       {cal_km:,.0f}')
print(f'Hierárquico Ward  {sil_hier:.4f}       {cal_hier:,.0f}')

# Scatter sobrepost
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for c in range(K_FINAL):
    mask = labels_km == c
    axes[0].scatter(X_comp[mask, 0], X_comp[mask, 1], s=3, alpha=0.3,
                    color=CORES_CLUSTER[c], label=f'C{c}')
axes[0].set_title(f'K-Means (Silhouette={sil_km:.4f})')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].legend(fontsize=8, markerscale=3)

for c in range(K_FINAL):
    mask = labels_hier == c
    axes[1].scatter(X_comp[mask, 0], X_comp[mask, 1], s=3, alpha=0.3,
                    color=CORES_CLUSTER[c], label=f'C{c}')
axes[1].set_title(f'Hierárquico Ward (Silhouette={sil_hier:.4f})')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].legend(fontsize=8, markerscale=3)

plt.suptitle(f'K-Means vs Hierárquico Ward — K={K_FINAL}', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'graficos' / '11_kmeans_vs_hierarquico.png', bbox_inches='tight')
plt.show()
print('→ Métricas similares entre algoritmos confirmam que a partição é robusta.')

In [ ]:
# ============================================================
# CÉLULA 15 — Análise de Estabilidade (20 Seeds)
# ============================================================
# Valida que os clusters não são artefatos da inicialização.
# Desvio padrão < 0,005 é considerado estável.

N_SEEDS   = 20
AMOSTRA_S = min(15_000, len(X_pca))
idx_stab  = np.random.RandomState(42).choice(len(X_pca), AMOSTRA_S, replace=False)
X_stab    = X_pca[idx_stab]

sil_seeds, inercia_seeds = [], []
print(f'Executando {N_SEEDS} seeds (K={K_FINAL}, amostra={AMOSTRA_S:,})...')

for seed in range(N_SEEDS):
    km_s = KMeans(n_clusters=K_FINAL, init='k-means++', n_init=5, random_state=seed)
    lbs  = km_s.fit_predict(X_stab)
    s    = silhouette_score(X_stab, lbs)
    sil_seeds.append(s)
    inercia_seeds.append(km_s.inertia_)
    print(f'  Seed {seed:2d}: Silhouette={s:.4f} | Inércia={km_s.inertia_:,.0f}')

media_sil = np.mean(sil_seeds)
std_sil   = np.std(sil_seeds)
print(f'\nSilhouette — Média: {media_sil:.4f} | Desvio padrão: {std_sil:.5f}')
print(f'→ {"ESTÁVEL" if std_sil < 0.005 else "ATENÇÃO — variabilidade alta"} (limiar < 0.005)')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(range(N_SEEDS), sil_seeds, marker='o', color='crimson')
axes[0].axhline(media_sil, color='red', linestyle='--',
                label=f'Média={media_sil:.4f} ± {std_sil:.5f}')
axes[0].set_title('Silhouette Score por Seed')
axes[0].set_xlabel('Seed'); axes[0].set_ylabel('Silhouette')
axes[0].legend()
axes[0].set_xticks(range(N_SEEDS))

axes[1].plot(range(N_SEEDS), inercia_seeds, marker='o', color='steelblue')
axes[1].axhline(np.mean(inercia_seeds), color='steelblue', linestyle='--',
                label=f'Média={np.mean(inercia_seeds):,.0f}')
axes[1].set_title('Inércia por Seed')
axes[1].set_xlabel('Seed'); axes[1].set_ylabel('Inércia')
axes[1].legend()
axes[1].set_xticks(range(N_SEEDS))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

plt.suptitle(f'Estabilidade do K-Means — {N_SEEDS} seeds, K={K_FINAL}', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'graficos' / '12_estabilidade_seeds.png', bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CÉLULA 16 — Exportação de Artefatos + PostgreSQL
# ============================================================

# 1. Parquet com cluster
df_features.to_parquet(OUTPUT_DIR / 'gestante_clusterizada.parquet', index=False)
print(f'✓ gestante_clusterizada.parquet ({len(df_features):,} registros)')

# 2. Centroides em escala original
centroides_df.to_csv(OUTPUT_DIR / 'centroides_originais.csv')
print('✓ centroides_originais.csv')

# 3. Modelo K-Means
joblib.dump(kmeans, OUTPUT_DIR / 'kmeans_maternar.pkl')
print('✓ kmeans_maternar.pkl')

# 4. PCA
joblib.dump(pca_full, OUTPUT_DIR / 'pca_maternar.pkl')
print('✓ pca_maternar.pkl')

# 5. Salvar cluster na tabela PostgreSQL via ADD COLUMN + UPDATE em lotes usando rownum
print('\nSalvando clusters no PostgreSQL...')
try:
    conn = psycopg2.connect(**DB_CONFIG)
    with conn.cursor() as cur:
        # Garantir coluna cluster
        cur.execute("""
            ALTER TABLE ml_maternar.gestante_features
            ADD COLUMN IF NOT EXISTS cluster SMALLINT;
        """)
        # Adicionar coluna rownum temporária para referência de linha
        cur.execute("""
            ALTER TABLE ml_maternar.gestante_features
            ADD COLUMN IF NOT EXISTS _rownum SERIAL;
        """)
        conn.commit()

        # Atualizar em lotes por rownum
        PAGE = 5000
        clusters_list = df_features['cluster'].astype(int).tolist()
        total = len(clusters_list)
        for start in range(0, total, PAGE):
            batch = [(clusters_list[i], start + i + 1)
                     for i in range(min(PAGE, total - start))]
            execute_values(
                cur,
                """UPDATE ml_maternar.gestante_features AS gf
                   SET cluster = v.c
                   FROM (VALUES %s) AS v(c, rn)
                   WHERE gf._rownum = v.rn""",
                batch,
            )
        conn.commit()

        # Remover coluna auxiliar
        cur.execute("""
            ALTER TABLE ml_maternar.gestante_features DROP COLUMN IF EXISTS _rownum;
        """)
        conn.commit()
    conn.close()
    print(f'✓ {total:,} registros atualizados em ml_maternar.gestante_features')
except Exception as e:
    print(f'⚠ PostgreSQL: {e}')

print('\n=== Artefatos exportados ===')
for f in sorted(OUTPUT_DIR.glob('*')):
    if f.is_file():
        print(f'  {f.name}')

print('\n=== Nomes clínicos dos clusters ===')
for c, nome in CLUSTER_LABELS.items():
    n = int((labels == c).sum())
    print(f'  {nome}: {n:,} gestantes ({n/len(labels)*100:.1f}%)')

---

# Relatório Final — KDD Maternar

## 1. Resumo Executivo

O pipeline KDD aplicado a **378.969 gestantes** (SISVAN 2014–2016, 20 features) gerou **4 clusters clinicamente interpretáveis** de perfis de risco gestacional, validados por múltiplas métricas e algoritmos independentes.

---

## 2. Dataset e Features

| Fonte | Tipo | Features |
|-------|------|----------|
| SISVAN | Individual | IMC, IMC pré-gestacional, ganho de IMC, peso, altura |
| SISVAN | Categórico OHE | Estado nutricional (4 flags), Raça/cor (5 flags) |
| SISVAN | Ordinal | Escolaridade (anos de estudo, 5 níveis) |
| SINAN | Municipal | log(1 + taxa sífilis gestacional / 1.000 consultas) |
| SIA | Municipal | log(1 + consultas pré-natais aprovadas) |
| CNES | Municipal | Número de hospitais/maternidades no município |
| Pipeline | Flag | flag_anti_hiv, tem_dado_sia |

---

## 3. Decisão de K=4

A escolha de K=4 prioriza **utilidade clínica** sobre otimalidade geométrica:

- **K=2**: distingue apenas risco/sem risco — insuficiente para triagem diferenciada
- **K=3**: não separa o risco municipal (infeccioso) do risco individual (nutricional)
- **K=4**: permite identificar 4 perfis acionáveis pelo app Maternar
- **K≥5**: fragmentação — clusters < 5% têm baixa representatividade estatística

---

## 4. Perfis dos Clusters (interpretar após ver os centroides)

| Cluster | Nome sugerido | Características esperadas | Prioridade |
|---------|---------------|---------------------------|------------|
| C0 | A definir | Ver heatmap de centroides | A definir |
| C1 | A definir | Ver heatmap de centroides | A definir |
| C2 | A definir | Ver heatmap de centroides | A definir |
| C3 | A definir | Ver heatmap de centroides | A definir |

> **Nota:** Os perfis devem ser nomeados após análise do heatmap (Célula 9) e boxplots (Célula 10), com validação de especialista clínico.

---

## 5. Validação Técnica

| Técnica | Resultado | Interpretação |
|---------|-----------|---------------|
| Elbow Method | Cotovelo em K=3-4 | Confirma K=4 |
| Silhouette Score | Ver Célula 8 | Moderado: sobreposição natural em dados clínicos |
| Calinski-Harabász | Ver Célula 8 | >500 = clusters bem separados |
| Dendrograma Ward | 4 grupos naturais | Corrobora K=4 independentemente |
| DBSCAN (ruído) | < 15% | Estrutura real, não ruído puro |
| Estabilidade (20 seeds) | σ < 0.005 | Solução robusta |
| K-Means vs Hierárquico | Silhouettes similares | Partição não é viés algorítmico |

---

## 6. Uso no App Maternar

O modelo salvo (`kmeans_maternar.pkl` + `pca_maternar.pkl` + `scaler_maternar.pkl`) permite classificar **novas gestantes em tempo real** na API Flask:

```python
import joblib, numpy as np

scaler = joblib.load('preprocess_output/scaler_maternar.pkl')
pca    = joblib.load('clustering_output/pca_maternar.pkl')
kmeans = joblib.load('clustering_output/kmeans_maternar.pkl')

def classificar_gestante(features_dict: dict) -> int:
    X = np.array([[features_dict[c] for c in COLS_ESCALADAS]])
    X_scaled = scaler.transform(X)
    X_pca    = pca.transform(X_scaled)
    return int(kmeans.predict(X_pca)[0])
```

---

## 7. Limitações

1. **Cobertura temporal**: dados de 2014–2016; perfis epidemiológicos podem ter mudado (pandemia, epidemia sífilis pós-2016)
2. **nu_semana_gestacional ausente**: sem dado de semana gestacional no CSV (SISVAN), impossível estratificar por trimestre
3. **CNES sem leitos obstétricos**: coluna `qt_leito_obs` nula no banco; feature usa apenas contagem de estabelecimentos
4. **Validação clínica pendente**: perfis precisam de revisão por obstetra para confirmar utilidade diagnóstica
5. **Imputação municipal**: 4% das gestantes tiveram taxas municipais imputadas com mediana (flag `tem_dado_sia=0`)

---

*Gerado por pipeline KDD — Projeto Maternar | `KDD_Maternar.ipynb`*